# Gaussian Naive Bayes

Load Pandas, plotting and [Sklearn](https://scikit-learn.org/stable/) packages.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn import datasets
from sklearn import metrics
from sklearn.naive_bayes import GaussianNB
from sklearn import preprocessing

## Tennis Example

In [ ]:
temp =  [ 54, 63, 81, 72, 68, 59, 83, 99, 66, 42, 103, 98, 80 ]
humidity = [ 40, 42, 60, 28, 70, 40, 43, 67, 20, 35, 70, 75, 30 ]
play = [ 1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 0, 0, 1 ]

In [ ]:
#Combine weather and temp into data frame
my_dict = {'Temp': temp,
           'Humidity': humidity,
           'Play': play,
           }
features = pd.DataFrame(my_dict)
features

In [ ]:
features[ ["Humidity"] ]

In [ ]:
features[["Humidity","Temp"]]

In [ ]:
# Create data sets
predictors = features[["Humidity","Temp"]]  # <= data frame
target = features["Play"]  # <= data series

# Initialize a GNB model
model = GaussianNB()

# Fit the model
model.fit(predictors, target) ;


In [ ]:
# Predict
input = pd.DataFrame( {'Humidity': [70], 'Temp': [100]} )

predicted = model.predict( input )
print("Predicted Value:", ("No","Yes")[predicted[0]]) # 0: No, 1: Yes

In [ ]:
probabilities = model.predict_proba(input)
(probabilities[0]*1000).astype(int) / 10

In [ ]:
# Predict
input = pd.DataFrame( {'Humidity': [35], 'Temp': [70]} )

predicted = model.predict( input )
print("Predicted Value:", ("No","Yes")[predicted[0]]) # 0: No, 1: Yes

In [ ]:
probabilities = model.predict_proba(input)
(probabilities*1000).astype(int) / 10

In [ ]:
probabilities

## Iris Example

Use dataset "Iris" from the Sklearn package.

In [ ]:
dataset = datasets.load_iris()

Examine data from dataset "Iris".

In [ ]:
# Look at features
iris_data = pd.DataFrame(dataset.data, columns=dataset.feature_names)
iris_data.head()

In [ ]:
iris_data.shape

In [ ]:
# Look at response
y = dataset.target
print(y)
print(dataset.target_names)

Plot histograms of predictors

In [ ]:
fig, axs = plt.subplots(nrows = 2, ncols = 2, figsize = (8,10))
colors = ['red', 'green', 'yellow', 'blue']
n = 0
for i in range(2):
  for j in range(2):
    column = dataset.feature_names[n]
    axs[i,j].hist(iris_data[column], color = colors[n], bins=30)
    axs[i,j].set_xlabel(column)
    n += 1

Plot scatter plots of certain predictors

In [ ]:
sns.scatterplot(x = iris_data['sepal length (cm)'], y = iris_data['sepal width (cm)'], hue = y, size = y)
plt.xlabel(dataset.feature_names[0])
plt.ylabel(dataset.feature_names[1])

In [ ]:
sns.scatterplot(x = iris_data['petal length (cm)'], y = iris_data['petal width (cm)'], hue = y, size = y)
plt.xlabel(dataset.feature_names[2])
plt.ylabel(dataset.feature_names[3])

Create a correlation plot of predictors

In [ ]:
sns.heatmap?


In [ ]:
column_correlations = iris_data.corr()
plt.figure(figsize=(8,8))
sns.heatmap(
    column_correlations,
    annot=True,
    xticklabels = dataset.feature_names,
    yticklabels = dataset.feature_names,
    vmin = -1,
    vmax = 1,
    cmap='BuPu',
);

Note that many of the predictors are correlated. GNB assumes the predictors are uncorrelated, but it is generally robust even if they are correlated. Let's see how it performs.

### Gaussian Naive Bayes
Define a Gaussian Naive Bayes model and use the data to train it.

In [ ]:
# Initialize a GNB model
model = GaussianNB()

In [ ]:
iris_data

In [ ]:
# Fit the model
model.fit(iris_data,y) ;

### Predictions for the same training set

In [ ]:
predicted_y = model.predict(iris_data)

In [ ]:
predicted_y

In [ ]:
y

In [ ]:
(predicted_y - y) * 10000

In [ ]:
(len(y)-6)/len(y)*100

In [ ]:
pd.Series(y).value_counts()

### Confusion matrix for the results.

In [ ]:
plt.figure(figsize=(8,8))
mat = metrics.confusion_matrix(y,predicted_y)
sns.heatmap(mat.T,
  square=True,
  annot=True,
  fmt='d',
  cbar=False,
  xticklabels=dataset.target_names,
  yticklabels=dataset.target_names
)
plt.xlabel('true label')
plt.ylabel('predicted label');

Calculate the classification accuracy.

In [ ]:
metrics.accuracy_score(y,predicted_y)*100


### Cross Validation


#### Split the training data into training set and testing set.

In [ ]:
import sklearn.model_selection as model_selection

X_train, X_test, y_train, y_test = model_selection.train_test_split(iris_data,y,test_size=0.25)


#### Verify

Verify the training and test data sizes are consistent with the original training data set.

In [ ]:
150* 0.75, 150* 0.25

In [ ]:
X_train.shape

In [ ]:
y_train.shape

In [ ]:
X_test.shape

In [ ]:
y_test.shape

### Cross-validation

In [ ]:
# One way
n = 100
results = np.zeros(n)
results

for idx in range(n):
  X_train, X_test, y_train, y_test = model_selection.train_test_split(iris_data,y,test_size=0.25)
  model = GaussianNB()
  model.fit(X_train,y_train)
  y_pred = model.predict(X_test)
  results[idx] = metrics.accuracy_score(y_test,y_pred)

print(results.mean())


In [ ]:
type(results)

In [ ]:
pd.Series(results).plot(kind="box")

In [ ]:
pd.Series(results).plot(kind="hist", bins=30)

In [ ]:
results.min(), results.max(), results.mean()

In [ ]:
results

In [ ]:
# Easier way
from sklearn.model_selection import cross_val_score
results = cross_val_score(model, iris_data, y, scoring='accuracy', cv = 10)
acc = results.mean()*100
acc


In [ ]:
iris_data.value_counts()